In [ ]:
import sys
import os
import numpy as  np
from itertools import combinations
import networkx as nx
import matplotlib.pyplot as plt


# Get the root directory (QAOA_mtrix_simulation)
sys.path.insert(0, r'C:\Users\N1259534\OneDrive - Nottingham Trent University\Desktop\Projects\QAOA_simulation_LV\QAOA_mtrix_simulation')
sys.executable


'c:\\Users\\N1259534\\AppData\\Local\\anaconda3\\envs\\qaoa_sim\\python.exe'

In [ ]:
np.random.seed(10)
terms = [(np.random.normal(), spin_pair) for spin_pair in combinations(range(N), r=2)]

### Test QAOA core component

In [3]:
# Simple 2-node graph
G_simple = nx.Graph()
G_simple.add_edge(0, 1, weight=1.0)

from main.Base.maxcut import get_maxcut_terms
# Get MaxCut terms
terms = get_maxcut_terms(G_simple)
print(f"Number of terms: {len(terms)}")
print("Terms (coefficient, qubits):")
for i , (coeff, qubits) in enumerate(terms):
    print(f" {i}: coeff= {coeff}, qubits={qubits}")

# Test with your current graph
print( f"\n\n graph G: {G_simple.number_of_nodes()} nodes, {G_simple.number_of_edges()} edge")

terms_g = get_maxcut_terms(G_simple)
print(f" generated {len(terms_g)} terms")

Number of terms: 2
Terms (coefficient, qubits):
 0: coeff= -0.5, qubits=(0, 1)
 1: coeff= 0.5, qubits=()


 graph G: 2 nodes, 1 edge
 generated 2 terms


In [4]:
from main.Base import choose_simulator
import numpy as np

# Simple problem
N = 2
terms_simple = [(-0.5, (0, 1))] 
print(f"Terms is : {terms_simple}")
print("\n")

simmulator = choose_simulator('auto')

sim = simmulator(N, terms = terms_simple)

#get cost diogonal 
cost_diag = sim.get_cost_diagonal()
print(f" The cost diogonal is : {cost_diag}")
print("\n")

for i, cost in enumerate(cost_diag):
    binary = format(i, f'0{N}b')
    print(f"  |{binary}⟩: {cost:8.4f}")

assert len(cost_diag) == 2**N, f" cost diagonal should have {2**N} elements"
print(f"\n✓ Cost diagonal has correct size: {len(cost_diag)}")

Terms is : [(-0.5, (0, 1))]


 The cost diogonal is : [-0.5  0.5  0.5 -0.5]


  |00⟩:  -0.5000
  |01⟩:   0.5000
  |10⟩:   0.5000
  |11⟩:  -0.5000

✓ Cost diagonal has correct size: 4


In [ ]:
# TEST 2.2: Test with your actual graph G
from main.QAOA_objective import get_qaoa_objective
import numpy as np

obj = get_qaoa_objective(G = G_simple, simulator= "python")



In [14]:
# Test single evaluation (p=2)
theta0 = np.array([0.1, 0.2, 0.3, 0.4])
print(f"\nTest theta: {theta0}")

cost_value = obj(theta0)
print(f"Cost Value: {cost_value:.8f}")

# Test multiple evaluations (for optimization)
print("\nTesting 5 random parameter sets:")
for i in range(5):
    theta_rand = np.random.rand(4)
    cost = obj(theta_rand)
    print(f"  Iteration {i+1}: theta={theta_rand}, cost={cost:.6f}")

print("\n✓ Objective function stable over multiple calls")


Test theta: [0.1 0.2 0.3 0.4]
Cost Value: -0.55815906

Testing 5 random parameter sets:
  Iteration 1: theta=[0.32551164 0.1650159  0.39252924 0.09346037], cost=-0.590293
  Iteration 2: theta=[0.82110566 0.15115202 0.38411445 0.94426071], cost=-0.314889
  Iteration 3: theta=[0.98762547 0.45630455 0.82612284 0.25137413], cost=-0.371141
  Iteration 4: theta=[0.59737165 0.90283176 0.53455795 0.59020136], cost=-0.508503
  Iteration 5: theta=[0.03928177 0.35718176 0.07961309 0.30545992], cost=-0.593128

✓ Objective function stable over multiple calls


In [23]:
# TEST 2.3: Parameter handling
from main import parameter_utils

# Test theta parameterization (default)
theta = np.array([0.1, 0.2, 0.3, 0.4])  # p=2
print(f" te theta is {theta}")

gamma, beta = parameter_utils.convert_to_gamma_beta(theta, parameterization='theta')

print(f"Converted_gamma = {gamma}")
print(f"Converted_beta = {beta}")

theta_reconstructed = np.hstack([gamma, beta])
print(f"Reconstructed theta: {theta_reconstructed}")

if np.allclose(theta, theta_reconstructed):
    print("\n✓ Parameter conversion is consistent")
else:
    print(" is not same")


 te theta is [0.1 0.2 0.3 0.4]
Converted_gamma = [0.1 0.2]
Converted_beta = [0.3 0.4]
Reconstructed theta: [0.1 0.2 0.3 0.4]

✓ Parameter conversion is consistent


### Test Integeration 

In [25]:
import numba.cuda

# Simple graph
G_test = nx.Graph()
G_test.add_edge(0, 1, weight=1.0)
G_test.add_edge(1, 2, weight=1.0)

theta = np.array([0.5, 0.3]) # p =1

obj = get_qaoa_objective(G = G_test, simulator= "auto")
cost_value = obj(theta)
print(f"objective value of cpu result is : {cost_value:.10f}")

# gpu (if available)
if numba.cuda.is_available():
    obj_gpu = get_qaoa_objective(G = G_test, simulator = "gpu")
    cost_value_gpu = obj_gpu(theta)
    print(f"gpu result objective value: {cost_value:.10f}")

    if np.isclose(cost_value, cost_value_gpu):
        print("\n✓ GPU and CPU results are consistent")
    else:
        print("\n✗ GPU and CPU results differ")
else:
    print("\n GPU is not available ")


objective value of cpu result is : -1.2270059152

 GPU is not available 


In [26]:
# test optimisation loop 
from scipy.optimize import minimize
from main.QAOA_objective import get_qaoa_objective

# small graph for testing
P = 1
G_new = nx.Graph()
G_new.add_edges_from([(0, 1), (1, 2), (2, 0)])  # Triangle
print (f"graph : {G_new.number_of_nodes()} nodes, {G_new.number_of_edges()} edges")

# create objective
obj = get_qaoa_objective(G = G_new, simulator= "auto")

theta_new = np.random.rand(2*P)
print(f" theta is {theta_new}")
print(f"the initial cost value is {obj(theta_new):.6f}")

#iteration
iteration_count = [0]
def callback(xk):
    iteration_count[0] +=1
    if iteration_count[0] % 5 ==0:
        print(f"  Iteration {iteration_count[0]}: cost={obj(xk):.6f}")

# optimise
result = minimize(obj, theta_new, method= 'COBYLA', callback= callback,
                   options = {'maxiter': 50})

print(f"final theta {result.x}")
print(f"final cost value is {result.fun:.6f}")
print(f"success: {result.success}")
print(f"totall iteration: {iteration_count[0]}")

if result.fun < obj(theta_new):
    print("\n✓ Optimization improved the cost value")
else:
    print("\n✗ Optimization did not improve the cost value")


graph : 3 nodes, 3 edges
 theta is [0.33071931 0.7738303 ]
the initial cost value is -1.470644
  Iteration 5: cost=-1.729741
  Iteration 10: cost=-1.881980
  Iteration 15: cost=-1.954700
  Iteration 20: cost=-1.980384
  Iteration 25: cost=-1.988534
  Iteration 30: cost=-1.991094
  Iteration 35: cost=-1.995728
  Iteration 40: cost=-1.998030
  Iteration 45: cost=-1.999153
  Iteration 50: cost=-1.999567
final theta [1.19168728 1.88324542]
final cost value is -1.999471
success: False
totall iteration: 51

✓ Optimization improved the cost value


### Data validation  (FPGA ready)



In [29]:
from main.Base.maxcut import get_maxcut_terms
from main.Base import choose_simulator

N = 2
G_fpga = nx.Graph()
G_fpga.add_edge(0, 1, weight=1.0)

# get terms and simulator 
terms = get_maxcut_terms(G_fpga)[:-1]

print(f"term is {terms}" )

sim = choose_simulator(name='python')(n_qubits=N, terms=terms)
cost_diagonal = sim.get_cost_diagonal()

for i, cost in enumerate(cost_diagonal):
    print(f"  [{i}]: {cost:.10f}  (type: {type(cost).__name__})")

# Check data types
assert all(isinstance(c, (float, np.floating)) for c in cost_diagonal), \
    "All costs should be floats"
print("✓ All costs are float type")


# Check size
assert len(cost_diagonal) == 2**N, f"Should have {2**N} costs"
print(f"✓ Correct size: {len(cost_diagonal)}")

# Initial state
initial_state = np.ones(2**N, dtype=np.complex128) / np.sqrt(2**N)
print(f"\nInitial state:")
for i, amp in enumerate(initial_state):
    print(f"  [{i}]: {amp.real:.6f} + {amp.imag:.6f}j")


# Parameters
gamma = np.array([0.5])
beta = np.array([0.3])
print(f"\nParameters:")
print(f"  gamma: {gamma} (type: {gamma.dtype})")
print(f"  beta:  {beta} (type: {beta.dtype})")
print(f"  cos(β): {np.cos(beta[0]):.10f}")
print(f"  sin(β): {np.sin(beta[0]):.10f}")


term is [(-0.5, (0, 1))]
  [0]: -0.5000000000  (type: float64)
  [1]: 0.5000000000  (type: float64)
  [2]: 0.5000000000  (type: float64)
  [3]: -0.5000000000  (type: float64)
✓ All costs are float type
✓ Correct size: 4

Initial state:
  [0]: 0.500000 + 0.000000j
  [1]: 0.500000 + 0.000000j
  [2]: 0.500000 + 0.000000j
  [3]: 0.500000 + 0.000000j

Parameters:
  gamma: [0.5] (type: float64)
  beta:  [0.3] (type: float64)
  cos(β): 0.9553364891
  sin(β): 0.2955202067


### Test FPGA part

**Data need to feed fpga**

1- Cost Diogonal: a array of energy vale for each computational basic state

    type: float64
    size: 2^N
    Target place: BRAM[2] (BRAM_COST_FUNC)

2- Initial state vector (BRAM[0] and BRAM[1])

    type: complex128 (real and imaginary)
    size: 2^N complex numbers
    Target place:  Real part: BRAM[0] (BRAM_STATE_REAL)
                   Imaginary: BRAM[1] (BRAM_STATE_IMAG)

3- QAOA Parameters (BRAM[5])

    Gamma and cos(β) sin(β) - 3xP (3 values per layer)
    Format: float64
    for 




